**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**11 | Multi-Seed Stability and Class-Imbalance Robustness**

![HARTH multi-seed and loss-robustness workflow](../../images/11_consolidated_workflow.svg)

**Research question:** How sensitive are the 1D-ResNet and TCN HARTH classifiers to model initialisation, and do alternative training losses change performance on minority activities under matched participant-held-out Development evaluation?

**Source cohort:** Official pinned HARTH v2.0 recordings; six lower-back/thigh acceleration channels at nominal 50 Hz; eight documented activity classes; 100-sample, single-label and timestamp-continuous windows with a 50-sample stride. The label-independent 500-window-per-person policy from [Notebook 10](10_harth_advanced_temporal_architectures_and_matched_benchmark.ipynb) yields **14,201 Development windows from 29 participants**, with S023/S032 excluded. The same source window counts and GroupKFold participant membership are checked against the current Notebook 10 CSV outputs before fitting.

**Experimental protocol:** Three seeds (42, 123, 2026) and three participant-disjoint folds for each of four conditions: 1D-ResNet and TCN with inverse-square-root weighted Cross-Entropy; 1D-ResNet with Focal Loss (gamma 2); and 1D-ResNet with Class-Balanced Cross-Entropy (beta 0.9999). This gives **12 unique model–loss–seed conditions and 36 model–fold jobs**, without duplicating the weighted-loss baseline. Each neural fit uses **400 Adam updates**, batch size **128** and learning rate **0.001**. Fold-specific sensor scaling and loss weights use Train participants only.

**Evaluation and outputs:** Fixed-eight-class Macro F1, balanced accuracy, paired seed/fold differences, class-wise precision/recall/F1 (including Shuffling and stair activities), participant-level variation and measured training/inference time. Current output files use the **`11_`** prefix and are saved to the study `output/` directory; local model–fold JSON caches are computational intermediates, not published evidence files.

**Scientific boundary:** Development-only, offline classification of overlapping activity windows. Repeated seeds assess limited initialisation sensitivity rather than independent replication or statistical superiority. This notebook does not train on reserved Test participants, select the final model or produce a qualified Test result.

**00. Colab Repository Setup**

In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")

if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**

In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("wearable_activity_recognition")

# ----------------- 2. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 3. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure

plot_style(
    font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
    label_size=11.5, label_weight="normal", tick_size=11,
    legend_size=10, figure_dpi=100, figure_size=(9, 4),
)

# ------------------ 4. Deep Learning and Research Utilities ------------------
import torch
from core.output_download import download_outputs_from_colab
from harth_data import (
    HARTH_SUBJECTS, HARTH_REVISION, ACTIVITY_NAMES, SENSOR_COLUMNS,
    prepare_harth_window_cohort,
)
from core.temporal_benchmark import make_benchmark_plan
from core.robustness_experiments import (
    ROBUST_MODELS, LOSS_METHODS, run_robustness_job,
    summarise_robustness_results, calculate_loss_weights,
)

# ----------- 5. Device Selection and Fixed Experimental Parameters -----------
from core.computer_device import computer_device
device = computer_device()
DEVICE = device.type
SEEDS = (42, 123, 2026)
CLASSES = tuple(range(1, 9))
RESERVED_TEST = {"S023", "S032"}
DEVELOPMENT = sorted(set(HARTH_SUBJECTS) - RESERVED_TEST)
FOLDS = 3
MAX_WINDOWS_PER_SUBJECT = 500
MAX_STEPS = 400
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
FOCAL_GAMMA = 2.0
BETA = 0.9999
CACHE_DIR = root / "data/processed/harth_stage24_robustness"

# ------------------------- 6. Study Output Directory -------------------------
output_dir = root / "wearable_activity_recognition" / "output"
output_dir.mkdir(parents=True, exist_ok=True)
print('HARTH revision:', HARTH_REVISION)
print('Development participants:', len(DEVELOPMENT))
print('Model initialisation seeds:', SEEDS)
print('Research outputs:', output_dir)
print('Model-fold cache (local):', CACHE_DIR)


**02. Reconstruct and Align the Matched Development Cohort**

Prepare the 29-person Development cohort with the same HARTH source reader, eight classes, window/stride settings and label-blind per-person sampling as Notebook 10. The source reader returns windows, labels and participant metadata in aligned row order. GroupKFold creates the same three participant-disjoint validation sets.

Two current Notebook 10 CSVs are **read-only reference inputs**: `10_matched_fold_design.csv` provides the held-out participant IDs for each fold, and `10_development_activity_support.csv` provides eight-class window counts. The checks compare observed identifiers and counts rather than asserting a historical total. A discrepancy would invalidate a matched cross-stage comparison and therefore stops the calculation. The benchmark's prediction scores are not re-used as newly computed robustness results.

In [ ]:
# ========= 02. Reconstruct and Align the Matched Development Cohort ==========

# ---------------- 1. Prepare the Shared HARTH Source Windows -----------------
X, y, metadata, overview = prepare_harth_window_cohort(
    subjects=DEVELOPMENT,
    data_dir=root / "data/raw/harth_v2",
    task_classes=CLASSES,
    max_windows_per_subject=MAX_WINDOWS_PER_SUBJECT,
    window_size=100,
    step_size=50,
    sampling_rate=50,
)
if X.ndim != 3 or X.shape[1:] != (len(SENSOR_COLUMNS), 100):
    raise ValueError(f"Unexpected HARTH window shape: {X.shape}.")
if len(X) != len(y) or len(y) != len(metadata):
    raise ValueError("Source windows, labels and participant rows differ.")
if set(metadata["subject_id"]) != set(DEVELOPMENT):
    raise ValueError("Development participant identities are incomplete.")
if set(metadata["subject_id"]) & RESERVED_TEST:
    raise ValueError("Reserved Test participants appear in Development data.")
if not np.isfinite(X).all() or not np.isin(y, CLASSES).all():
    raise ValueError("Non-finite sensor data or an unknown task class.")

# ------------- 2. Preserve Participant-Disjoint Model Evaluation -------------
plan = make_benchmark_plan(
    X, y, metadata["subject_id"].to_numpy(), CLASSES,
    n_splits=FOLDS, forbidden_subjects=RESERVED_TEST,
)
for fold, train_rows, validation_rows in plan["folds"]:
    training_people = set(plan["groups"][train_rows])
    validation_people = set(plan["groups"][validation_rows])
    if training_people & validation_people:
        raise ValueError(f"Fold {fold} contains overlapping participants.")

# -------------- 3. Check Current Notebook 10 Reference Evidence --------------
fold_path = output_dir / "10_matched_fold_design.csv"
support_path = output_dir / "10_development_activity_support.csv"
for source_path in (fold_path, support_path):
    if not source_path.is_file():
        raise FileNotFoundError(
            f"Missing required Notebook 10 evidence: {source_path}"
        )
benchmark_folds = pd.read_csv(fold_path).set_index("fold")
benchmark_support = pd.read_csv(support_path).set_index("activity")
if set(benchmark_folds.index) != {f for f, _, _ in plan["folds"]}:
    raise ValueError("Notebook 10 and 11 use different fold identifiers.")
for fold, train_rows, validation_rows in plan["folds"]:
    observed_people = set(plan["groups"][validation_rows])
    recorded_people = {
        item.strip()
        for item in str(benchmark_folds.loc[fold, "validation_subject_ids"]).split(",")
        if item.strip()
    }
    if observed_people != recorded_people:
        raise ValueError(f"Fold {fold}: held-out participants differ from Notebook 10.")
    if len(validation_rows) != int(benchmark_folds.loc[fold, "validation_windows"]):
        raise ValueError(f"Fold {fold}: held-out window count differs from Notebook 10.")

activity_support = pd.Series({
    ACTIVITY_NAMES[code]: int(np.sum(y == code)) for code in CLASSES
})
recorded_support = benchmark_support["windows"].reindex(activity_support.index)
if recorded_support.isna().any() or not np.array_equal(
    activity_support.to_numpy(), recorded_support.to_numpy()
):
    raise ValueError("The eight activity window counts differ from Notebook 10.")

# ------------- 4. Present and Export Development Source Context --------------
print("Development participants:", metadata["subject_id"].nunique())
print("Source window tensor:", X.shape)
print("GroupKFold validation windows:", [
    len(validation_rows) for _, _, validation_rows in plan["folds"]
])
print("Source fingerprint:", plan["fingerprint"])
display(pd.DataFrame({
    "activity": activity_support.index,
    "windows": activity_support.to_numpy(),
}))
overview.to_csv(output_dir / "11_development_cohort_summary.csv", index=False)


**03. Define the Model, Loss, Seed and Fold Grid**

The experiment contains four distinct architecture/loss conditions. Each is evaluated at seeds 42, 123 and 2026 and on all three matched participant folds. The two weighted Cross-Entropy conditions (1D-ResNet and TCN) are each trained only once per seed/fold; the Focal and Class-Balanced alternatives are additional ResNet conditions. Thus, the full grid has **12 unique model–loss–seed combinations** and **36 model–fold jobs**, with no duplicated baseline fit.

Parameters are held fixed across conditions, apart from the intended architecture or loss change. Save the complete 36-row job grid and a compact protocol table, while displaying the four distinct conditions rather than a redundant listing of every job.

In [ ]:
# ============== 03. Define the Model, Loss, Seed and Fold Grid ===============

# ----------- 1. Declare Four Nonduplicated Experimental Conditions -----------
MODEL_LOSS_CONDITIONS = (
    ("1D-ResNet", "weighted_ce"),
    ("TCN", "weighted_ce"),
    ("1D-ResNet", "focal"),
    ("1D-ResNet", "class_balanced"),
)
if any(architecture not in ROBUST_MODELS or loss not in LOSS_METHODS
       for architecture, loss in MODEL_LOSS_CONDITIONS):
    raise ValueError("An unsupported model or loss has been configured.")
EXPERIMENTS = [
    (architecture, loss, seed)
    for architecture, loss in MODEL_LOSS_CONDITIONS
    for seed in SEEDS
]
if len(EXPERIMENTS) != len(set(EXPERIMENTS)):
    raise ValueError("Duplicate model–loss–seed conditions.")

job_grid = pd.DataFrame([
    {"architecture": architecture, "loss": loss, "seed": seed, "fold": fold}
    for architecture, loss, seed in EXPERIMENTS
    for fold, _, _ in plan["folds"]
])
display(pd.DataFrame([
    {"architecture": model, "loss_method": method,
     "seeds": len(SEEDS), "folds_per_seed": FOLDS}
    for model, method in MODEL_LOSS_CONDITIONS
]))
print("Distinct model–loss–seed conditions:", len(EXPERIMENTS))
print("Total model–fold jobs:", len(job_grid))

# ---------------- 2. Record the Shared Optimisation Settings -----------------
conditions = pd.DataFrame([
    {"parameter": "Development window cap", "value": MAX_WINDOWS_PER_SUBJECT},
    {"parameter": "Adam updates per fit", "value": MAX_STEPS},
    {"parameter": "Batch size", "value": BATCH_SIZE},
    {"parameter": "Learning rate", "value": LEARNING_RATE},
    {"parameter": "Focal gamma", "value": FOCAL_GAMMA},
    {"parameter": "Class-balanced beta", "value": BETA},
    {"parameter": "Metric", "value": "Fixed eight-class Macro F1"},
])
job_grid.to_csv(output_dir / "11_robustness_job_grid.csv", index=False)
conditions.to_csv(output_dir / "11_robustness_conditions.csv", index=False)


**04. Inspect Training-Only Class Weights and Loss Definitions**

**Weighted Cross-Entropy:** Per-class weights are proportional to the inverse square root of each fold's Train-label count and are normalised to a mean weight of one.

**Focal Loss (gamma = 2):** Uses the same Train-derived weights as Weighted CE, scaled by the hard-example factor `(1 - p_true)^gamma` and divided by the sum of mini-batch weights.

**Class-Balanced CE (beta = 0.9999):** Uses effective-number weights calculated from each fold's Train-label counts. Its effective-count assumptions are descriptive approximations because 50%-overlapping windows are not independent observations.

The table illustrates weights from the **first Train fold only**; every fit recomputes weights from its own Train participants inside `run_robustness_job`. No held-out labels contribute to scaling or loss weighting.

In [ ]:
# ======= 04. Inspect Training-Only Class Weights and Loss Definitions ========

# ------------------ 1. Summarise First-Fold Training Labels ------------------
first_fold, first_train, first_validation = plan["folds"][0]
training_people = set(plan["groups"][first_train])
validation_people = set(plan["groups"][first_validation])
if training_people & validation_people:
    raise ValueError("Train and Validation participants overlap.")
train_counts = pd.Series(y[first_train]).value_counts().reindex(
    CLASSES, fill_value=0
)
loss_weights = pd.DataFrame({
    "class_code": CLASSES,
    "activity": [ACTIVITY_NAMES[code] for code in CLASSES],
    "train_windows": train_counts.to_numpy(),
})
for method in LOSS_METHODS:
    loss_weights[method] = calculate_loss_weights(
        y[first_train], CLASSES, method=method, beta=BETA
    )
display(loss_weights.round(4))
print("Displayed class weights belong to Train fold:", first_fold)
loss_weights.to_csv(output_dir / "11_fold_train_class_weights.csv", index=False)


**05. Fit or Restore the 36 Matched Robustness Jobs**

The experiment runs every predeclared architecture/loss/seed combination on the same three held-out participant folds. ResNet and TCN receive separate seeded initialisations. Each fit uses 400 Adam updates, batch size 128, learning rate 0.001, Train-only sensor standardisation and Train-only loss weighting.

This section requires CUDA. The existing local, Git-ignored source-fingerprinted cache in `data/processed/harth_stage24_robustness/` is used for safe restoration of compatible individual model–fold records, avoiding duplicate training when saved evidence is valid. Cache configuration includes code identity, source and row fingerprints, fold, seed, loss parameters and device. It is not itself part of the published `11_` CSV/PNG outputs.

In [ ]:
# ============= 05. Fit or Restore the 36 Matched Robustness Jobs =============

# ---------------------- 1. Set the Compute Environment -----------------------
if DEVICE != "cuda":
    raise RuntimeError("A CUDA runtime is required for the matched GPU study.")
torch.set_float32_matmul_precision("highest")
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

# -------------- 2. Evaluate All Matched Experimental Conditions --------------
jobs = []
for architecture, loss_method, seed in EXPERIMENTS:
    print(
        f"Architecture: {architecture} | Loss: {loss_method} | Seed: {seed}"
    )
    for fold, _, _ in plan["folds"]:
        result = run_robustness_job(
            plan, architecture, fold, seed, loss_method, CACHE_DIR,
            device=DEVICE, max_steps=MAX_STEPS,
            batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE,
            focal_gamma=FOCAL_GAMMA, beta=BETA,
        )
        jobs.append(result)

if len(jobs) != len(job_grid):
    raise RuntimeError("The model–loss–seed–fold experiment grid is incomplete.")
print("Completed/restored model–fold records:", len(jobs))


**06. Reconstruct Paired Out-of-Fold Evidence**

Reconstruct held-out predictions from the 36 saved model–fold records, retaining actual source-row identifiers, original labels, participant IDs, fold membership and all eight classes. The shared aggregation module returns fold-level scores, person-level fixed-eight-class Macro F1, pooled class-level precision/recall/F1 with support, and source-aligned predictions.

The large raw OOF tables are exported once without repeating them in notebook displays. Across 12 model–loss–seed conditions, each of the same Development windows should receive exactly one held-out prediction per condition. The final section checks complete source coverage and target alignment.

In [ ]:
# ================ 06. Reconstruct Paired Out-of-Fold Evidence ================

# --------------- 1. Aggregate the Held-Out Experiment Records ----------------
fold_results, subject_results, class_results, predictions = (
    summarise_robustness_results(plan, jobs, EXPERIMENTS)
)
print("Fold-level result rows:", len(fold_results))
print("Participant-level result rows:", len(subject_results))
print("Class-level result rows:", len(class_results))
print("Source-row predictions:", len(predictions))

# ---------------- 2. Write Full-Precision Robustness Evidence ----------------
fold_results.to_csv(output_dir / "11_robustness_fold_metrics.csv", index=False)
subject_results.to_csv(output_dir / "11_robustness_subject_metrics.csv", index=False)
class_results.to_csv(output_dir / "11_robustness_class_metrics.csv", index=False)
predictions.to_csv(output_dir / "11_robustness_oof_predictions.csv", index=False)


**07. Compare ResNet and TCN Across Seeds**

Keep inverse-square-root Weighted Cross-Entropy fixed and compare 1D-ResNet with TCN at all three seeds using the same three participant-held-out folds. The resulting six architecture/seed summaries contain fold-mean Macro F1 and within-seed fold standard deviation.

The plotted three-point seed profiles describe initialisation sensitivity for this source cohort. They are not confidence intervals or a substitute for independent population replication. Full fold-level evidence remains in the `11_robustness_fold_metrics.csv` export.

In [ ]:
# ================== 07. Compare ResNet and TCN Across Seeds ==================

# ------------------------- 1. Compare ResNet and TCN -------------------------
stability=fold_results.loc[fold_results["loss_method"]=="weighted_ce"]
seed_summary=stability.groupby(["architecture","seed"],as_index=False).agg(
    mean_macro_f1=("macro_f1","mean"),
    fold_sd=("macro_f1","std"),
)
display(seed_summary.round(4))
seed_summary.to_csv(output_dir / "11_weighted_ce_seed_summary.csv", index=False)

# ------------------------ 2. Summarise Seed Stability ------------------------
fig,ax=plt.subplots(figsize=(10,5))
for arch,frame in seed_summary.groupby("architecture"):
    ax.plot(frame["seed"].astype(str),frame["mean_macro_f1"],marker="o",label=arch)
ax.set(xlabel="Reproducibility seed",ylabel="Mean three-fold Macro F1",
       title="Weighted CE | ResNet and TCN across seeds",ylim=(0,1))
ax.grid(alpha=0.2)
ax.legend()
fig.tight_layout()
save_figure(fig, output_dir / "11_architecture_seed_stability.png")
plt.show()


**08. Compare Paired ResNet Loss Functions**

Compare Weighted Cross-Entropy, Focal Loss and Class-Balanced Cross-Entropy using the same architecture, seed, participant fold, source rows and update budget. For each seed/fold pair, the table includes the three measured Macro F1 values and paired differences from Weighted CE. The corresponding plot makes gains and losses visible without selecting the best seed.

This comparison isolates the loss formulation within the present ResNet pipeline. It does not establish a universal advantage of one loss for every participant or activity.

In [ ]:
# ================= 08. Compare Paired ResNet Loss Functions ==================

# --------------------- 1. Compare Matched ResNet Losses ----------------------
resnet=fold_results[fold_results["architecture"]=="1D-ResNet"]
paired_loss=resnet.pivot(index=["seed","fold"],columns="loss_method",values="macro_f1")
paired_loss["focal_minus_weighted"]=paired_loss["focal"]-paired_loss["weighted_ce"]
paired_loss["class_balanced_minus_weighted"]=(
    paired_loss["class_balanced"]-paired_loss["weighted_ce"]
)
display(paired_loss.round(4))

# --------------------- 2. Export Paired Loss Comparisons --------------------
paired_loss.reset_index().to_csv(output_dir / "11_resnet_paired_loss_scores.csv", index=False)

# ----------------------- 3. Summarise Loss Sensitivity -----------------------
fig,ax=plt.subplots(figsize=(11,5))
paired_loss[[
    "focal_minus_weighted","class_balanced_minus_weighted"
]].plot(kind="bar",ax=ax)
ax.axhline(0,color="black",linewidth=.8)
ax.set(title="ResNet | Paired loss changes relative to Weighted CE",
       xlabel="Seed and held-out fold",ylabel="Macro F1 difference")
ax.grid(axis="y",alpha=0.2)
fig.tight_layout()
save_figure(fig, output_dir / "11_resnet_loss_ablation.png")
plt.show()


**09. Assess Shuffling and Stair-Activity Performance**

Analyse pooled held-out class-level precision, recall, F1 and actual support for Shuffling and both stair activities under every model–loss–seed condition. Rare-class metrics use the same original Development cohort and are not resampled or balanced at evaluation time.

The figure isolates Shuffling F1 for the ResNet loss comparison across the three seeds. An apparent rare-class improvement must be interpreted alongside precision, recall and fixed-eight-class Macro F1: increasing recall for one activity may coincide with losses elsewhere.

In [ ]:
# ============ 09. Assess Shuffling and Stair-Activity Performance ============

# ------------------- 1. Inspect Minority-Class Performance -------------------
minority=class_results.loc[class_results["class_code"].isin([3,4,5])].copy()
minority["activity"]=minority["class_code"].map(ACTIVITY_NAMES)
minority.to_csv(output_dir / "11_minority_activity_metrics.csv", index=False)
display(minority[[
    "architecture","loss_method","seed","activity",
    "support","precision","recall","f1"
]].round(4))

# ------------------ 2. Compare Shuffling and Stair Outcomes ------------------
shuffle=minority.loc[
    (minority["architecture"]=="1D-ResNet") & (minority["class_code"]==3)
]
fig,ax=plt.subplots(figsize=(11,5))
for method,frame in shuffle.groupby("loss_method"):
    ax.plot(frame["seed"].astype(str),frame["f1"],marker="o",label=method)
ax.set(xlabel="Random seed",ylabel="Pooled out-of-fold Shuffling F1",
       title="ResNet | Shuffling F1 by loss and seed",ylim=(0,1))
ax.grid(alpha=0.2)
ax.legend()
fig.tight_layout()
save_figure(fig, output_dir / "11_shuffling_loss_sensitivity.png")
plt.show()


**10. Summarise Participant Variation and Compute Cost**

Participant-level Macro F1 is calculated over the fixed eight activity classes, irrespective of an individual's observed class support. The participant summary reports within-condition mean and SD across people; the compute summary reports fold-mean and fold-SD Macro F1 plus total recorded training and inference duration.

Save the two distinct research tables, but display a single joined 12-row summary for legibility. Training times refer to the recorded model fits and depend on the actual hardware; restoration from an existing cache does not constitute new model training.

In [ ]:
# =========== 10. Summarise Participant Variation and Compute Cost ============

# -------------------- 1. Compare Participant-Level Scores --------------------
people_summary=subject_results.groupby(
    ["architecture","loss_method","seed"],as_index=False
).agg(
    mean_person_macro_f1=("macro_f1_fixed_task","mean"),
    person_macro_f1_sd=("macro_f1_fixed_task","std")
)

# ------------------------ 2. Summarise Training Cost -------------------------
compute=fold_results.groupby(
    ["architecture","loss_method","seed"],as_index=False
).agg(
    mean_fold_macro_f1=("macro_f1","mean"),
    fold_macro_f1_sd=("macro_f1","std"),
    total_train_s=("train_seconds","sum"),
    total_inference_s=("inference_seconds","sum"),
)

# -------------------- 3. Display Distinct Research Summaries --------------------
display(people_summary.merge(
    compute, on=["architecture", "loss_method", "seed"],
    validate="one_to_one"
).round(4))

# ------------------- 4. Export Person and Compute Results -------------------
people_summary.to_csv(output_dir / "11_person_level_robustness.csv", index=False)
compute.to_csv(output_dir / "11_training_compute_summary.csv", index=False)


**11. Interpret Repeated-Seed and Loss Robustness**

Rank the four architecture/loss conditions by the mean of their three seed-specific, three-fold mean Macro F1 scores. The accompanying across-seed SD refers to variation among three **seed means**; it is not a population confidence interval. Selection of a final protocol is a separate analysis using current Development evidence.

Important limitations include the small seed set, correlated overlapping windows, class-balanced effective-number assumptions, unequal model family capacity, post-hoc comparisons across conditions and earlier descriptive inspection of the reserved S023/S032 recordings in the wider study. The present experiment neither re-evaluates those participants nor provides independent external validation.

In [ ]:
# ============== 11. Interpret Repeated-Seed and Loss Robustness ==============

# -------------------- 1. Interpret Development Robustness --------------------
final_summary=compute.groupby(
    ["architecture","loss_method"],as_index=False
).agg(
    mean_across_seed_fold_macro_f1=("mean_fold_macro_f1","mean"),
    across_seed_sd=("mean_fold_macro_f1","std"),
    total_training_s=("total_train_s","sum"),
)
display(final_summary.sort_values(
    "mean_across_seed_fold_macro_f1",ascending=False
).round(4))

# ----------------------- 2. State Evidence Limitations -----------------------
limitations=pd.DataFrame([
    {"issue":"Development-only estimates","boundary":"No independent held-out Test claim"},
    {"issue":"Three repeated seeds","boundary":"Limited stochastic robustness evidence"},
    {"issue":"Overlapping source windows","boundary":"Windows are not independent observations"},
    {"issue":"Approximate class counts","boundary":"Class-balanced effective number assumes counts as proxies"},
    {"issue":"Loss selection","boundary":"Select on Development only; disclose post-hoc comparisons"},
    {"issue":"Original Test historical inspection","boundary":"S023/S032 not strictly preregistered-unseen"},
])

# ----------------------- 3. Export Robustness Evidence ----------------------
final_summary.to_csv(output_dir / "11_robustness_model_summary.csv", index=False)
limitations.to_csv(output_dir / "11_robustness_limitations.csv", index=False)


**12. Verify Source Alignment and Package Research Outputs**

Verify that all specified model–loss–seed combinations cover the three source-matched participant folds, that no condition duplicates or omits held-out source rows, that true labels match the original Development data and that no reserved Test participants appear in saved predictions. Check valid fixed-task F1 ranges and report expected/observed row counts.

These scientific checks protect the credibility of the exported Development evidence without introducing an administrative approval flag. The stage-specific download includes only `11_` CSV/PNG outputs. Final model selection and qualified held-out evaluation are outside Notebook 11.

In [ ]:
# ========= 12. Verify Source Alignment and Package Research Outputs ==========

# -------------- 1. Check Complete, Paired Experimental Evidence --------------
expected_conditions = len(EXPERIMENTS)
expected_jobs = expected_conditions * len(plan["folds"])
expected_predictions = expected_conditions * len(y)
if len(jobs) != expected_jobs or len(fold_results) != expected_jobs:
    raise ValueError("Missing model–fold result records.")
if fold_results.duplicated(
    ["architecture", "loss_method", "seed", "fold"]
).any():
    raise ValueError("Duplicated model–loss–seed–fold result.")
if len(subject_results) != expected_conditions * len(DEVELOPMENT):
    raise ValueError("Incomplete participant-level robustness evidence.")
if len(class_results) != expected_conditions * len(CLASSES):
    raise ValueError("Incomplete eight-class robustness evidence.")
if len(predictions) != expected_predictions:
    raise ValueError("Incomplete source-row out-of-fold predictions.")
for key, table in predictions.groupby(
    ["architecture", "loss_method", "seed"]
):
    if len(table) != len(y) or not table["source_row"].is_unique:
        raise ValueError(f"Source-row coverage differs for {key}.")
    if set(table["subject_id"]) & RESERVED_TEST:
        raise ValueError(f"Reserved Test participant appears in {key}.")
    observed_truth = table.sort_values("source_row")["true_label"].to_numpy()
    if not np.array_equal(observed_truth, y):
        raise ValueError(f"Original target alignment differs for {key}.")
if not fold_results["macro_f1"].between(0, 1).all():
    raise ValueError("Invalid fixed-eight-class Macro F1 values.")

# ------------- 2. Describe the Evidence and Download Stage Files -------------
print("Development participants:", len(DEVELOPMENT))
print("Source windows per condition:", len(y))
print("Model–loss–seed conditions:", expected_conditions)
print("Model–fold records (expected / observed):",
      expected_jobs, "/", len(fold_results))
print("Held-out predictions (expected / observed):",
      expected_predictions, "/", len(predictions))
print("Activity-level results (expected / observed):",
      expected_conditions * len(CLASSES), "/", len(class_results))
print("Research output directory:", output_dir)

download_outputs_from_colab(
    folder="wearable_activity_recognition/output", stage="11"
)
